# Ensemble Learning for Breast Cancer Classification

## Project Overview

This project demonstrates ensemble learning using a real breast cancer classification dataset. The goal is to predict whether a tumor is malignant or benign using several ensemble machine learning methods.

Ensemble learning combines multiple models to produce stronger and more stable predictions than a single model. This project compares bagging, random forests, extra trees, boosting, voting classifiers, and stacking classifiers.

## Dataset

The dataset contains diagnostic measurements from breast tumor cell nuclei. Each observation represents a tumor sample, and the features describe characteristics such as radius, texture, perimeter, area, smoothness, compactness, concavity, and symmetry.

For this project, the target is recoded so that:

- `1 = malignant`
- `0 = benign`

This makes malignant tumors the positive class, which is useful because detecting malignant tumors is usually the more important medical objective.

## Models Compared

- Decision Tree
- Bagging Classifier
- Random Forest
- Extra Trees
- AdaBoost
- Gradient Boosting
- Histogram Gradient Boosting
- Voting Classifier
- Stacking Classifier

## Evaluation Metrics

The models are evaluated using accuracy, precision, recall, specificity, F1 score, ROC AUC, average precision, and log loss.

## Ensemble Methods Explained

### Bagging

Bagging trains many models on different random samples of the training data and averages their predictions. It reduces variance and helps prevent overfitting.

### Random Forest

Random Forest is a bagging method that builds many decision trees. Each tree sees a random sample of the data and a random subset of features. This usually performs better than a single decision tree.

### Extra Trees

Extra Trees is similar to Random Forest, but it adds more randomness when choosing split points. This can reduce variance and improve generalization.

### Boosting

Boosting builds models sequentially. Each new model tries to correct the mistakes of the previous models. AdaBoost and Gradient Boosting are common boosting methods.

### Voting Classifier

A Voting Classifier combines different models and makes predictions based on the majority vote or average predicted probabilities.

### Stacking Classifier

Stacking trains several base models, then uses another model called a meta model to learn how to combine their predictions.

### Import libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

### Load the dataset

In [4]:
df = pd.read_csv("../data/breast_cancer_classification.csv")

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis,diagnosis_label
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0,malignant


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [8]:
df["diagnosis_label"].value_counts()

diagnosis_label
benign       357
malignant    212
Name: count, dtype: int64

In [9]:
print("0 = malignant\n1 = benign")

0 = malignant
1 = benign


### Recode data

Recode the target so that 1 = malignant and 0 = benign

In [10]:
X = df.drop(columns=["diagnosis", "diagnosis_label"])

y = np.where(df["diagnosis_label"] == "malignant", 1, 0)

y = pd.Series(y, name="malignant")

In [12]:
y.value_counts()

malignant
0    357
1    212
Name: count, dtype: int64

In [13]:
y.value_counts(normalize=True)

malignant
0    0.627417
1    0.372583
Name: proportion, dtype: float64

### Train test split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20,
    stratify=y, random_state=42 )

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Training data shape: (455, 30)
Test data shape: (114, 30)

Training class distribution:
malignant
0    0.626374
1    0.373626
Name: proportion, dtype: float64

Test class distribution:
malignant
0    0.631579
1    0.368421
Name: proportion, dtype: float64


### Define the models

In [16]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10, random_state=42),

    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42), n_estimators=200,
        max_samples=0.8, bootstrap=True, random_state=42, n_jobs=-1),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=3, random_state=42, n_jobs=-1),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=3, random_state=42, n_jobs=-1),

    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=200,
        learning_rate=0.05, random_state=42),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),

    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=200, max_leaf_nodes=31, random_state=42)
}